# PowerOps_v1 — Notebook 04: Metadata-Aware Retrieval

This is the notebook that makes PowerOps a **hybrid** retrieval system rather
than a plain semantic-search demo. Notebook 03 showed pure vector similarity
working, but similarity scores were only moderate (~0.38–0.46) — this dataset
has short, terse Summaries, and there's nothing semantically special about a
team name or a status value that would make embeddings reliably surface it.

So structured parts of a question (team, assignee, priority, status, issue
key) get handled by **exact Pinecone metadata filters**, and whatever's left
gets handled by semantic search on the remainder of the question.

### Design decisions carried over from Notebook 01

- The parser's vocabulary comes from `data/vocabulary.json` — the **actual**
  distinct teams/priorities/statuses/assignees in this dataset — not
  hardcoded guesses.
- Priority matching is **literal only**: this dataset has no "Critical" tier
  (only `Low`/`Medium`/`High`/`Urgent`), so a question asking for "critical
  issues" will correctly find **no** priority filter rather than guessing
  that "critical" means "Urgent." The gap is real and should surface as
  reduced precision, not be silently papered over.
- Same principle for status: there's no literal "Open"/"Blocked"/"Resolved"
  in this data (only `To Do`/`In Progress`/`Done`/`Rejected`/`On Hold`/`Soft
  Delete`), so those words won't match a status filter either.


## 1. Load configuration, vocabulary, and reconnect to Pinecone

In [1]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "powerops-v1")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
TOP_K = int(os.environ.get("TOP_K", "5"))

with open("../data/vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

print("Teams:     ", vocabulary["assigned_teams"])
print("Priorities:", vocabulary["priorities"])
print("Statuses:  ", vocabulary["statuses"])
print(f"Assignees: {len(vocabulary['assignees'])} distinct")


Teams:      ['Falcon Squad', 'Nova Team', 'Summit Crew']
Priorities: ['High', 'Low', 'Medium', 'Urgent']
Statuses:   ['Done', 'In Progress', 'On Hold', 'Rejected', 'Soft Delete', 'To Do']
Assignees: 62 distinct


In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

stats = index.describe_index_stats()
print(f"Connected to '{PINECONE_INDEX_NAME}' — {stats['total_vector_count']} vectors.")


Connected to 'powerops-v1' — 1000 vectors.


## 2. Issue key detection

`Issue key` values follow a strict `INO-<digits>` pattern, so this is a simple
regex — but note it's deliberately strict: a question asking about `OPS-1024`
(a different prefix, not present in this dataset) should **not** match,
because it genuinely refers to an issue that doesn't exist here.


In [3]:
ISSUE_KEY_PATTERN = re.compile(r"\bINO-\d+\b", re.IGNORECASE)

def find_issue_key(question: str) -> str | None:
    match = ISSUE_KEY_PATTERN.search(question)
    return match.group(0).upper() if match else None


print(find_issue_key("What is the status of issue INO-21920?"))
print(find_issue_key("What is the status of issue OPS-1024?"))


INO-21920
None


## 3. Team, priority, and status detection

Each of these is matched as a whole phrase, case-insensitively, directly
against the real vocabulary values — no synonyms, no fuzzy matching. Statuses
are checked longest-first so a two-word status like `On Hold` isn't
accidentally partially matched by a shorter one.


In [4]:
def find_vocab_match(question: str, values: list[str]) -> str | None:
    """Return the first vocabulary value found as a whole-phrase match in the question."""
    q_lower = question.lower()
    for value in sorted(values, key=len, reverse=True):
        pattern = r"\b" + re.escape(value.lower()) + r"\b"
        if re.search(pattern, q_lower):
            return value
    return None


test_questions = [
    "What high priority issues are assigned to Falcon Squad?",
    "Show me all Urgent issues for Nova Team",
    "Show me all critical issues.",
    "What Rejected issues does Summit Crew have?",
    "What issues are currently On Hold?",
]
for q in test_questions:
    print(f"{q!r}")
    print(f"  team:     {find_vocab_match(q, vocabulary['assigned_teams'])}")
    print(f"  priority: {find_vocab_match(q, vocabulary['priorities'])}")
    print(f"  status:   {find_vocab_match(q, vocabulary['statuses'])}")


'What high priority issues are assigned to Falcon Squad?'
  team:     Falcon Squad
  priority: High
  status:   None
'Show me all Urgent issues for Nova Team'
  team:     Nova Team
  priority: Urgent
  status:   None
'Show me all critical issues.'
  team:     None
  priority: None
  status:   None
'What Rejected issues does Summit Crew have?'
  team:     Summit Crew
  priority: None
  status:   Rejected
'What issues are currently On Hold?'
  team:     None
  priority: None
  status:   On Hold


## 4. Assignee detection — and why it's the hardest part

`Assignee` is stored as `"Last, First (Contractor)"`. Your example questions
use bare first names ("John", "Sarah") — and this dataset simply doesn't
contain those people (Notebook 01's profiling showed 62 real assignees, none
named John or Sarah). More importantly, several *real* first names repeat
across different people — there are three different people named `Anjali`
(`Gibson, Anjali`; `Rice, Anjali (Contractor)`; `Porter, Anjali (Contractor)`).

So the assignee matcher needs two tiers:

1. **Full name match** (both first and last name present in the question) —
   unambiguous, highest confidence.
2. **Single-token match** (only a first or last name given) — if it uniquely
   identifies one assignee, use it; if it matches *multiple* people, that's
   a genuine ambiguity PowerOps should surface rather than silently guessing
   one of them.


In [5]:
def _split_name(assignee: str) -> tuple[str, str]:
    """Split 'Last, First (Contractor)' into (last, first)."""
    cleaned = assignee.replace("(Contractor)", "").strip()
    if "," in cleaned:
        last, first = (p.strip() for p in cleaned.split(",", 1))
    else:
        parts = cleaned.split()
        last, first = (parts[0], " ".join(parts[1:])) if parts else ("", "")
    return last, first


NAME_PARTS = {a: _split_name(a) for a in vocabulary["assignees"]}


def find_assignee(question: str) -> dict:
    """Match an assignee against the question.

    Returns a dict:
      {"assignee": <name or None>, "ambiguous_candidates": [<names>] or None}
    """
    q_lower = question.lower()

    # Tier 1: full name match (both first and last name present)
    full_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        if not last or not first:
            continue
        if (re.search(r"\b" + re.escape(last.lower()) + r"\b", q_lower)
                and re.search(r"\b" + re.escape(first.lower()) + r"\b", q_lower)):
            full_matches.add(assignee)

    if len(full_matches) == 1:
        return {"assignee": next(iter(full_matches)), "ambiguous_candidates": None}
    if len(full_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(full_matches)}

    # Tier 2: single-token match (first name OR last name, length >= 3 to avoid noise)
    token_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        for token in (first, last):
            if token and len(token) >= 3 and re.search(r"\b" + re.escape(token.lower()) + r"\b", q_lower):
                token_matches.add(assignee)
                break

    if len(token_matches) == 1:
        return {"assignee": next(iter(token_matches)), "ambiguous_candidates": None}
    if len(token_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(token_matches)}

    return {"assignee": None, "ambiguous_candidates": None}


for q in [
    "What issues does Gibson, Anjali have?",
    "What issues does Anjali have?",
    "What open issues does John have?",
    "What critical issues are assigned to Sarah?",
]:
    print(f"{q!r} -> {find_assignee(q)}")


'What issues does Gibson, Anjali have?' -> {'assignee': 'Gibson, Anjali', 'ambiguous_candidates': None}
'What issues does Anjali have?' -> {'assignee': None, 'ambiguous_candidates': ['Gibson, Anjali', 'Porter, Anjali (Contractor)', 'Rice, Anjali (Contractor)']}
'What open issues does John have?' -> {'assignee': None, 'ambiguous_candidates': None}
'What critical issues are assigned to Sarah?' -> {'assignee': None, 'ambiguous_candidates': None}


## 5. `parse_query_filters()` — combine everything into one deterministic parser

This assembles issue-key/team/priority/status/assignee detection into a
single function, and also computes the **semantic remainder** — the question
with every matched phrase stripped out, used for the vector-similarity part
of retrieval. If nothing was matched (or stripping empties the question), the
full original question is used as the semantic query.


In [6]:
def parse_query_filters(question: str) -> dict:
    """Deterministically extract structured filters from a PowerOps question.

    Returns:
        {
            "filters": {"assigned_team": ..., "priority": ..., "status": ...,
                        "issue_key": ..., "assignee": ...}  # only keys that matched
            "ambiguous_assignee_candidates": [...] or None,
            "semantic_query": "...",
        }
    """
    filters = {}
    matched_spans = []

    issue_key = find_issue_key(question)
    if issue_key:
        filters["issue_key"] = issue_key
        matched_spans.append(issue_key)

    team = find_vocab_match(question, vocabulary["assigned_teams"])
    if team:
        filters["assigned_team"] = team
        matched_spans.append(team)

    priority = find_vocab_match(question, vocabulary["priorities"])
    if priority:
        filters["priority"] = priority
        matched_spans.append(priority)

    status = find_vocab_match(question, vocabulary["statuses"])
    if status:
        filters["status"] = status
        matched_spans.append(status)

    assignee_result = find_assignee(question)
    if assignee_result["assignee"]:
        filters["assignee"] = assignee_result["assignee"]
        # Strip whichever name token(s) actually matched, roughly.
        last, first = NAME_PARTS[assignee_result["assignee"]]
        matched_spans.extend([t for t in (last, first) if t])

    semantic_query = question
    for span in matched_spans:
        semantic_query = re.sub(re.escape(span), "", semantic_query, flags=re.IGNORECASE)
    semantic_query = re.sub(r"\s+", " ", semantic_query).strip(" ?.")
    if not semantic_query:
        semantic_query = question

    return {
        "filters": filters,
        "ambiguous_assignee_candidates": assignee_result["ambiguous_candidates"],
        "semantic_query": semantic_query,
    }


import json as _json
print(_json.dumps(parse_query_filters("What high priority issues are assigned to Falcon Squad?"), indent=2))


{
  "filters": {
    "assigned_team": "Falcon Squad",
    "priority": "High"
  },
  "ambiguous_assignee_candidates": null,
  "semantic_query": "What priority issues are assigned to"
}


## 6. Optional: LLM-based structured-output parsing (for comparison only)

As an alternative to the deterministic parser, an LLM can be asked to
produce the same structured output — constrained to the real vocabulary via
the system prompt, and using LangChain's `with_structured_output` so the
result is validated against a Pydantic schema rather than freeform text.

**This is a comparison demonstration, not what `retrieve_powerops_documents`
uses.** The deterministic parser above is fully predictable, has zero
per-query LLM cost/latency, and can never invent a team/priority/status that
doesn't exist in the data — all valuable properties for a POC that must
avoid hallucination. An LLM parser could in principle handle more varied
phrasing, but introduces exactly the failure mode PowerOps needs to guard
against: a model confidently returning a filter value that isn't real.


In [7]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

CHAT_MODEL = os.environ.get("CHAT_MODEL", "gpt-4o-mini")


class ParsedQuery(BaseModel):
    semantic_query: str = Field(description="The core semantic search text, with structured filter terms removed")
    assigned_team: str | None = Field(default=None, description="Must be one of the known teams, or null")
    priority: str | None = Field(default=None, description="Must be one of the known priorities, or null")
    status: str | None = Field(default=None, description="Must be one of the known statuses, or null")
    assignee: str | None = Field(default=None, description="Must be one of the known assignees (exact 'Last, First' form), or null")
    issue_key: str | None = Field(default=None, description="An INO-##### issue key if mentioned, or null")


_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
_structured_llm = _llm.with_structured_output(ParsedQuery)

_system_prompt = (
    "You extract structured filters from a DevOps question for PowerOps. "
    "Only use these exact known values (case-sensitive) - never invent new ones:\n"
    f"Teams: {vocabulary['assigned_teams']}\n"
    f"Priorities: {vocabulary['priorities']}\n"
    f"Statuses: {vocabulary['statuses']}\n"
    "If a concept in the question (e.g. 'critical', 'open', 'blocked') does not exactly match "
    "one of the known values above, leave that field null rather than guessing the closest one."
)


def parse_query_filters_llm(question: str) -> ParsedQuery:
    return _structured_llm.invoke([
        ("system", _system_prompt),
        ("human", question),
    ])


for q in [
    "What high priority issues are assigned to Falcon Squad?",
    "Show me all critical issues.",
]:
    deterministic = parse_query_filters(q)
    llm_result = parse_query_filters_llm(q)
    print(f"QUESTION: {q}")
    print(f"  deterministic filters: {deterministic['filters']}")
    print(f"  llm-parsed:            {llm_result.model_dump(exclude={'semantic_query'})}")
    print()


QUESTION: What high priority issues are assigned to Falcon Squad?
  deterministic filters: {'assigned_team': 'Falcon Squad', 'priority': 'High'}
  llm-parsed:            {'assigned_team': 'Falcon Squad', 'priority': 'High', 'status': None, 'assignee': None, 'issue_key': None}



QUESTION: Show me all critical issues.
  deterministic filters: {}
  llm-parsed:            {'assigned_team': None, 'priority': None, 'status': None, 'assignee': None, 'issue_key': None}



## 7. Build Pinecone filters and `retrieve_powerops_documents()`

Each detected filter becomes an exact-match (`$eq`) constraint. Multiple
filters combine as an implicit AND (Pinecone's default when a dict has
multiple top-level keys) — e.g. team **and** priority **and** status all at
once when a question specifies all three.


In [8]:
def build_pinecone_filter(filters: dict) -> dict:
    """Convert parsed filters into a Pinecone metadata filter expression."""
    pinecone_filter = {}
    for key in ("assigned_team", "assignee", "priority", "status", "issue_key"):
        if key in filters:
            pinecone_filter[key] = {"$eq": filters[key]}
    return pinecone_filter


def retrieve_powerops_documents(question: str, top_k: int = TOP_K) -> dict:
    """Hybrid retrieval: deterministic metadata filters + semantic vector search.

    Returns a dict with the parsed filters, the Pinecone filter actually applied,
    and the retrieved (document, score) pairs.
    """
    parsed = parse_query_filters(question)
    pinecone_filter = build_pinecone_filter(parsed["filters"])

    results = vector_store.similarity_search_with_score(
        parsed["semantic_query"],
        k=top_k,
        filter=pinecone_filter or None,
    )

    return {
        "question": question,
        "filters": parsed["filters"],
        "ambiguous_assignee_candidates": parsed["ambiguous_assignee_candidates"],
        "semantic_query": parsed["semantic_query"],
        "pinecone_filter": pinecone_filter,
        "results": results,
    }


## 8. Test at least 10 questions

A deliberate mix: questions that should produce clean filters, questions
that reference real-but-ambiguous names, and questions that reference things
that genuinely don't exist in this dataset (a person, a status word, an
issue-key prefix) — the honest "no match" behavior here is exactly what
Notebook 06's escalation logic will build on.


In [9]:
TEST_QUESTIONS = [
    "What high priority issues are assigned to Falcon Squad?",
    "Show me all Urgent issues for Nova Team",
    "What issues does Gibson, Anjali have?",
    "What issues does Anjali have?",
    "What is the status of issue INO-21920?",
    "What Rejected issues does Summit Crew have?",
    "Show me all critical issues.",
    "What open issues does John have?",
    "What are the To Do issues for Falcon Squad?",
    "What issues are currently On Hold?",
    "Summarize the open problems for Team Falcon Squad",
    "What is the status of issue OPS-1024?",
]

for question in TEST_QUESTIONS:
    result = retrieve_powerops_documents(question, top_k=5)
    print("=" * 78)
    print(f"QUESTION: {question}")
    print(f"  Detected filters:     {result['filters']}")
    if result["ambiguous_assignee_candidates"]:
        print(f"  Ambiguous assignees:  {result['ambiguous_assignee_candidates']}")
    print(f"  Semantic query used:  {result['semantic_query']!r}")
    print(f"  Pinecone filter:      {result['pinecone_filter']}")
    print(f"  Retrieved {len(result['results'])} document(s):")
    for doc, score in result["results"]:
        m = doc.metadata
        print(f"    [{score:.4f}] {m['issue_key']} | {m['assigned_team']} | {m['assignee']} | "
              f"priority={m['priority']} | status={m['status']}")


QUESTION: What high priority issues are assigned to Falcon Squad?
  Detected filters:     {'assigned_team': 'Falcon Squad', 'priority': 'High'}
  Semantic query used:  'What priority issues are assigned to'
  Pinecone filter:      {'assigned_team': {'$eq': 'Falcon Squad'}, 'priority': {'$eq': 'High'}}
  Retrieved 5 document(s):
    [0.4155] INO-20549 | Falcon Squad | Mason, Anthony | priority=High | status=Done
    [0.3951] INO-21149 | Falcon Squad | Mason, Ashley (Contractor) | priority=High | status=Done
    [0.3566] INO-20216 | Falcon Squad | Mason, Ashley (Contractor) | priority=High | status=Done
    [0.3558] INO-21550 | Falcon Squad | Mason, Ashley (Contractor) | priority=High | status=Done
    [0.3549] INO-21032 | Falcon Squad | Mason, Ashley (Contractor) | priority=High | status=Done


QUESTION: Show me all Urgent issues for Nova Team
  Detected filters:     {'assigned_team': 'Nova Team', 'priority': 'Urgent'}
  Semantic query used:  'Show me all issues for'
  Pinecone filter:      {'assigned_team': {'$eq': 'Nova Team'}, 'priority': {'$eq': 'Urgent'}}
  Retrieved 5 document(s):
    [0.2765] INO-21146 | Nova Team | Webb, Amy | priority=Urgent | status=Done
    [0.2598] INO-20065 | Nova Team | Myers, Deepa | priority=Urgent | status=Done
    [0.2592] INO-21839 | Nova Team | Myers, Deepa | priority=Urgent | status=Done
    [0.2539] INO-20283 | Nova Team | Myers, Deepa | priority=Urgent | status=Done
    [0.2477] INO-21115 | Nova Team | Prasad, Carolyn (Contractor) | priority=Urgent | status=Done


QUESTION: What issues does Gibson, Anjali have?
  Detected filters:     {'assignee': 'Gibson, Anjali'}
  Semantic query used:  'What issues does , have'
  Pinecone filter:      {'assignee': {'$eq': 'Gibson, Anjali'}}
  Retrieved 5 document(s):
    [0.2738] INO-20769 | Nova Team | Gibson, Anjali | priority=Medium | status=To Do
    [0.2700] INO-20176 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
    [0.2658] INO-20771 | Nova Team | Gibson, Anjali | priority=Medium | status=To Do
    [0.2646] INO-21851 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
    [0.2590] INO-21745 | Nova Team | Gibson, Anjali | priority=Medium | status=In Progress


QUESTION: What issues does Anjali have?
  Detected filters:     {}
  Ambiguous assignees:  ['Gibson, Anjali', 'Porter, Anjali (Contractor)', 'Rice, Anjali (Contractor)']
  Semantic query used:  'What issues does Anjali have'
  Pinecone filter:      {}
  Retrieved 5 document(s):
    [0.4553] INO-21689 | Nova Team | Porter, Anjali (Contractor) | priority=Medium | status=Soft Delete
    [0.4474] INO-21639 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done
    [0.4436] INO-21636 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done
    [0.4432] INO-21637 | Summit Crew | Tucker, Eric | priority=Medium | status=Done
    [0.4329] INO-21638 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done


QUESTION: What is the status of issue INO-21920?
  Detected filters:     {'issue_key': 'INO-21920'}
  Semantic query used:  'What is the status of issue'
  Pinecone filter:      {'issue_key': {'$eq': 'INO-21920'}}
  Retrieved 1 document(s):
    [0.4539] INO-21920 | Falcon Squad | Sullivan, Deepa (Contractor) | priority=Medium | status=In Progress


...

QUESTION: Show me all critical issues.
  Detected filters:     {}
  Semantic query used:  'Show me all critical issues'
  Pinecone filter:      {}
  Retrieved 5 document(s):
    [0.4922] INO-20547 | Nova Team | Myers, Deepa | priority=High | status=Done
    [0.4184] INO-21369 | Falcon Squad | Mason, Ashley (Contractor) | priority=Urgent | status=Done
    [0.4183] INO-21851 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
    [0.4131] INO-20176 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
    [0.4066] INO-21863 | Nova Team | Ford, Kenneth (Contractor) | priority=Medium | status=In Progress


QUESTION: What open issues does John have?
  Detected filters:     {}
  Semantic query used:  'What open issues does John have'
  Pinecone filter:      {}
  Retrieved 5 document(s):
    [0.4331] INO-20769 | Nova Team | Gibson, Anjali | priority=Medium | status=To Do
    [0.4218] INO-20754 | Nova Team | Webb, Amy | priority=Medium | status=Done
    [0.4035] INO-20755 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.3962] INO-19858 | Nova Team | Ellis, Anthony | priority=High | status=Done
    [0.3927] INO-21556 | Summit Crew | Wells, Jack (Contractor) | priority=Medium | status=In Progress
QUESTION: What are the To Do issues for Falcon Squad?
  Detected filters:     {'assigned_team': 'Falcon Squad', 'status': 'To Do'}
  Semantic query used:  'What are the issues for'
  Pinecone filter:      {'assigned_team': {'$eq': 'Falcon Squad'}, 'status': {'$eq': 'To Do'}}
  Retrieved 5 document(s):
    [0.3189] INO-21555 | Falcon Squad | Bose, Alexander | priority=

QUESTION: What issues are currently On Hold?
  Detected filters:     {'status': 'On Hold'}
  Semantic query used:  'What issues are currently'
  Pinecone filter:      {'status': {'$eq': 'On Hold'}}
  Retrieved 2 document(s):
    [0.2543] INO-21801 | Nova Team | Reyes, Catherine (Contractor) | priority=Medium | status=On Hold
    [0.2373] INO-21560 | Nova Team | Reyes, Catherine (Contractor) | priority=Medium | status=On Hold


QUESTION: Summarize the open problems for Team Falcon Squad
  Detected filters:     {'assigned_team': 'Falcon Squad'}
  Semantic query used:  'Summarize the open problems for Team'
  Pinecone filter:      {'assigned_team': {'$eq': 'Falcon Squad'}}
  Retrieved 5 document(s):
    [0.4224] INO-20136 | Falcon Squad | Mehta, Daniel | priority=Medium | status=Done
    [0.4177] INO-20021 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.4123] INO-20131 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.4117] INO-20135 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.4114] INO-20204 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done


QUESTION: What is the status of issue OPS-1024?
  Detected filters:     {}
  Semantic query used:  'What is the status of issue OPS-1024'
  Pinecone filter:      {}
  Retrieved 5 document(s):
    [0.5512] INO-20644 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done
    [0.5466] INO-20021 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.5376] INO-21874 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
    [0.5365] INO-20899 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done
    [0.5363] INO-20247 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done


## Summary & next steps

- Built a fully deterministic `parse_query_filters()` that detects issue
  keys (regex), teams/priorities/statuses (exact vocabulary match), and
  assignees (two-tier name matching that correctly flags ambiguity instead
  of guessing among same-first-name people).
- Confirmed the literal-only design decision behaves as intended: "critical"
  and "open"/"blocked" correctly produce **no** priority/status filter
  (since neither exists in this dataset's real vocabulary), rather than a
  fabricated mapping.
- Confirmed honest "not found" behavior for a person (`John`) and an
  issue-key prefix (`OPS-1024`) that don't exist in this data — exactly the
  signal Notebook 06's escalation logic needs.
- Demonstrated an optional LLM-based structured-output parser for
  comparison, constrained to the real vocabulary — but the deterministic
  parser remains the one `retrieve_powerops_documents()` actually uses, for
  predictability and zero hallucination risk in filter extraction.
- `retrieve_powerops_documents(question, top_k)` combines parsed filters
  with Pinecone metadata filtering and semantic search, tested against 12
  questions covering both clean matches and genuine gaps.

**Next: Notebook 05 — RAG Question Answering.** We'll wrap
`retrieve_powerops_documents()` with an LLM answer-generation step
(`ask_powerops()`), using a system prompt that grounds answers strictly in
retrieved context and cites `Issue Key`s as evidence.
